# Build technician knowledge-profile artefacts

Generates the bundled ``.npz`` knowledge grids that every templated technician (``junior``, ``senior``, ``generalist``, ``expert``, ``trainee``, ``motor_specialist``, ``electronics_specialist``) loads at episode start.  Each grid is a snapshot of an `ongoing.KnowledgeGrid` populated with synthetic experience: the **profile spec** controls *how much* career experience the technician walks in with and *where* on the embedding grid that experience sits.

**Output**: `src/kata/resources/technician_profiles/<name>.npz`, one per profile, plus a `profiles.txt` sidecar listing what was generated.

**When to re-run**:
- After editing ``default_profiles()`` in `kata.EntityFactories.technician_profile_builder`.
- After changing the knowledge-grid shape or hyperparameters in the corresponding technician template (the grid's saved shape must match the template's ``knowledge_k_shape``).

The build is deterministic from a fixed seed --- the same spec + seed produces byte-identical artefacts on any machine.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

# Anchor to the repo root so relative paths resolve consistently.
_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / "run_configs").is_dir():
    _root = _root.parent
if (_root / "run_configs").is_dir():
    os.chdir(_root)
    print(f"Working directory: {_root}")
else:
    raise RuntimeError(f"Could not locate repo root from {Path.cwd()}")

os.environ["KATA_CONF_PATH"] = "/dev/null/__no_file__"

import numpy as np
import matplotlib.pyplot as plt

from kata.EntityFactories.technician_profile_builder import (
    DEFAULT_OUTPUT_DIR,
    build_default_profiles,
    build_one,
    default_profiles,
)

print(f"Output dir: {DEFAULT_OUTPUT_DIR}")

## 2. Catalogue

List the profiles we're about to generate and their key parameters.

In [ ]:
specs = default_profiles()
print(f"{'name':<25s}{'shape':<10s}{'n_tickets':>12s}{'lr':>8s}{'sigma_prop':>14s}")
print('-' * 70)
for s in specs:
    print(
        f"{s.name:<25s}{str(s.grid_shape):<10s}{s.n_tickets:>12d}"
        f"{s.learning_rate:>8.2f}{s.propagation_sigma:>14.2f}"
    )
print()
for s in specs:
    print(f"  {s.name:<25s} {s.description}")

## 3. Build all profiles

Calls into the shared builder so this notebook stays in sync with the CI / programmatic pipeline.

In [ ]:
written = build_default_profiles(seed=0)
print(f"\nWrote {len(written)} grids:")
for p in written:
    print(f"  {p}")

## 4. Visualise each profile's knowledge map

Heatmap of the knowledge field for every generated profile.  Brighter cells = more accumulated expertise at that embedding location.  Specialists show a hotspot in their corner; generalists / experts show broad coverage.

In [ ]:
from ongoing import KnowledgeGrid

grids = [(s.name, s, KnowledgeGrid.load(DEFAULT_OUTPUT_DIR / f"{s.name}.npz")) for s in specs]
n = len(grids)
ncols = 4
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.2, nrows * 3.0))
axes = np.atleast_2d(axes).flatten()

# Use a common color scale so heatmaps are comparable across profiles.
vmax = max(g.get_max_knowledge() for _, _, g in grids)
vmax = max(vmax, 1e-6)

for ax, (name, spec, g) in zip(axes, grids):
    grid_arr = g._grid  # private but stable; ongoing exposes no public read API
    im = ax.imshow(grid_arr, cmap="viridis", vmin=0, vmax=vmax, origin="lower")
    ax.set_title(
        f"{name}\nmax_k={g.get_max_knowledge():.2f}  "
        f"spec_idx={g.specialisation_index():.2f}",
        fontsize=9,
    )
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
for ax in axes[len(grids):]:
    ax.axis("off")
fig.suptitle("Technician knowledge profiles", y=1.02, fontsize=12)
fig.tight_layout()
plt.show()

## 5. Per-profile statistics

A tabular summary so the visualisation can be cross-checked numerically.

In [ ]:
import pandas as pd

rows = []
for name, spec, g in grids:
    rows.append({
        "profile": name,
        "n_tickets": spec.n_tickets,
        "max_knowledge": float(g.get_max_knowledge()),
        "max_experiences": float(g.get_max_experiences()),
        "knowledge_volume": float(g.knowledge_volume()),
        "specialisation_index": float(g.specialisation_index()),
        "knowledge_entropy": float(g.knowledge_entropy()),
    })
pd.DataFrame(rows)

## 6. (Optional) Add a custom profile

If you need a profile that's not in the default catalogue, build it directly with `build_one(spec, seed=...)`, save manually, and add a matching entry to `technician_templates.json` pointing at the new file.

```python
from kata.EntityFactories.technician_profile_builder import ProfileSpec, build_one, gaussian_at

spec = ProfileSpec(
    name="pump_specialist",
    grid_shape=(10, 10),
    propagation_sigma=1.0, transmission_factor=0.5, learning_rate=0.6,
    n_tickets=600,
    embedding_sampler=gaussian_at((5.0, 2.0), sigma=1.0),
    description="Pump / hydraulic specialist (central-bottom of the grid).",
)
grid = build_one(spec, seed=0)
grid.save(DEFAULT_OUTPUT_DIR / f"{spec.name}.npz")
```